In [ ]:
import pandas as pd

df = pd.read_excel('../../../data/data_battery_materials_PRISTINE_CORRECTED_15.09.xlsx')
df.head()

In [ ]:
df_substrate = df[['smiles (substrate)', 'capacitance_max']].rename(columns={'smiles (substrate)': 'smiles'})
df_poreside = df[['smiles (pore side)', 'capacitance_max']].rename(columns={'smiles (pore side)': 'smiles'})
df_node = df[['SMILES to be used (molecules with a node)', 'capacitance_max']].rename(columns={'SMILES to be used (molecules with a node)': 'smiles'})

In [ ]:
from rdkit import Chem

valid_substrate = [i for i, s in enumerate(df_substrate['smiles']) if Chem.MolFromSmiles(s) is not None]
valid_poreside = [i for i, s in enumerate(df_poreside['smiles']) if Chem.MolFromSmiles(s) is not None]
valid_node = [i for i, s in enumerate(df_node['smiles']) if Chem.MolFromSmiles(s) is not None]

valid_intersection = list(set(valid_substrate) & set(valid_poreside) & set(valid_node))

print(valid_substrate)
print(valid_poreside)
print(valid_node)
print(len(df_substrate), len(valid_substrate), len(valid_poreside), len(valid_node))
print(valid_intersection)
print(len(valid_intersection))

## CFTs

In [ ]:
from rdkit.Chem import Draw

def find_nitrile_group(smiles_string):
  """
  Checks for the presence of a nitrile group (-CN) in a molecule
  represented by a SMILES string.

  Args:
    smiles_string: The SMILES string of the molecule.

  Returns:
    A tuple containing a boolean indicating if the group was found,
    and the atom indices of the matches.
  """
  mol = Chem.MolFromSmiles(smiles_string)
  if mol is None:
    print("Invalid SMILES string.")
    return False, []

  nitrile_smarts = "[C]#[N]"
  nitrile_pattern = Chem.MolFromSmarts(nitrile_smarts)
  has_nitrile = mol.HasSubstructMatch(nitrile_pattern)
  matches = mol.GetSubstructMatches(nitrile_pattern)
  return has_nitrile

def visualize_nitrile_groups_in_grid(smiles_list):
  """
  Generates a grid image of molecules with their nitrile (-CN) groups highlighted.
  This version correctly flattens the atom indices for highlighting.

  Args:
    smiles_list: A list of SMILES strings for the molecules to be visualized.
  """
  mols = [Chem.MolFromSmiles(smi) for smi in smiles_list if smi] # Filter out potential empty strings

  # SMARTS pattern for the nitrile group
  nitrile_smarts = "[C]#[N]"
  nitrile_pattern = Chem.MolFromSmarts(nitrile_smarts)

  highlight_atom_lists = []
  for mol in mols:
    if mol:
      # Get all matches for the nitrile pattern in the molecule
      matches = mol.GetSubstructMatches(nitrile_pattern)

      # *** FIX: Flatten the tuple of tuples into a single list of integers ***
      atoms_to_highlight = []
      for match in matches:
        atoms_to_highlight.extend(match)
      highlight_atom_lists.append(atoms_to_highlight)
    else:
      # This case might not be reached if filtering SMILES, but is good practice
      highlight_atom_lists.append([])

  # Generate the grid image
  display(Draw.MolsToGridImage(
      mols,
      molsPerRow=3,
      subImgSize=(400, 400),
      legends=[Chem.MolToSmiles(m) for m in mols],
      highlightAtomLists=highlight_atom_lists
  ))

In [ ]:
df_valid_substrates = df_substrate.iloc[valid_substrate]
df_valid_poreside = df_poreside.iloc[valid_poreside]
df_valid_node = df_node.iloc[valid_node]

In [ ]:
df_substrates_with_nitrile = df_valid_substrates[df_valid_substrates['smiles'].apply(find_nitrile_group)]

In [ ]:
print(f'CTFs: {len(df_substrates_with_nitrile)}')
print(f'Total: {len(df_valid_substrates)}')

In [ ]:
visualize_nitrile_groups_in_grid(df_substrates_with_nitrile['smiles'].tolist())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def remove_outliers(df):
    score = 'capacitance_max'
    Q1 = df[score].quantile(0.25)
    Q3 = df[score].quantile(0.75)
    IQR = Q3 - Q1

    # 3. Define the lower and upper bounds for outlier detection
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_no_outliers = df[~((df[score] < lower_bound) | (df[score] > upper_bound))]
    return df_no_outliers

print('CFTs')
df_no_outliers = remove_outliers(df_substrates_with_nitrile)
sns.histplot(df_no_outliers['capacitance_max'], kde=True, bins=20)
plt.show()
print('All')
df_no_outliers = remove_outliers(df_valid_substrates)
sns.histplot(df_no_outliers['capacitance_max'], kde=True, bins=50)
plt.show()

## Patterns in data

In [ ]:
from PIL import Image
import requests
from io import BytesIO

def visualize_smarts_rdkit(smarts):
  query_mol = Chem.MolFromSmarts(smarts)

  # Check if the SMARTS string was valid and could be parsed
  if query_mol is None:
      print(f"Error: Invalid SMARTS string provided: '{smarts}'")
      return None

  # Generate the image for the single query molecule.
  # We can use the SMARTS string itself as a legend for clarity.
  display(Draw.MolToImage(query_mol, size=(300, 300), legend=smarts))

def visualize_smarts_smartsplus(smarts, scale=0.5):
  query = smarts
  base_url = 'https://smarts.plus/smartsview/download_rest?smarts='
  query = query.replace('%', '%25')
  query = query.replace('&', '%26')
  query = query.replace('+', '%2B')
  query = query.replace('#', '%23')
  query = query.replace(';', '%3B')

  url = base_url + query
  res = requests.get(url)
  image = Image.open(BytesIO(res.content))
  new_size = (int(image.width * scale), int(image.height * scale))
  image = image.resize(new_size)
  display(image)

def visualize_smarts_with_smiles(smiles_list, smarts):
  """
  Counts the number of SMILES that match a SMARTS pattern and displays a grid of the matching molecules with the pattern highlighted.

  Args:
      smarts: A SMARTS string representing the substructure pattern.
      smiles_list: A list of SMILES strings representing the molecules to be searched.

  Returns:
      The number of molecules that match the SMARTS pattern.
  """
  pattern = Chem.MolFromSmarts(smarts)
  if pattern is None:
      raise ValueError("Invalid SMARTS pattern provided.")

  matching_mols = []
  for smiles in smiles_list:
      mol = Chem.MolFromSmiles(smiles)
      if mol and mol.HasSubstructMatch(pattern):
          matching_mols.append(mol)

  if not matching_mols:
      print("No molecules matched the provided SMARTS pattern.")
      return 0

  highlight_atom_lists = [mol.GetSubstructMatch(pattern) for mol in matching_mols]
  img = Draw.MolsToGridImage(matching_mols,
                              molsPerRow=3,
                              subImgSize=(400, 400),
                              legends=[Chem.MolToSmiles(m) for m in matching_mols],
                              highlightAtomLists=highlight_atom_lists)

  # display(img)

  return len(matching_mols)

### Visualize smarts patterns

In [ ]:
smarts_patterns = [
  '[#6]1=[#6]2[#6]([#6]=[#6][#6]=[#6]2)=[#6][#6]=[#6]1',
  '[#6]1=[#6]2[#6]([#6]=[#6][#6]2)=[#6][#6]=[#6]1',
  '[#6]@[#6](@[#6])(@[#6])',
  '[#6]1=[#6]([#6]2=[#6][#6]=[#6][#6]=[#6]2)[#6]=[#6][#6]=[#6]1',
  '[#6]1=[#6][#6]=[#6][#7]=[#6]1',
  '[#6]1=[#6][#7]=[#6][#7]=[#6]1',
  '[#7]1=[#6][#7]=[#6][#7]=[#6]1',
  '[#6]1=[#6][#6]=[#16][#6]1',
  '[#6](=[#7]-[#6]=*)(-[#6]=*)-[#7](-[#6]=*)(-[#6]=*)',
  '[#6]1[#6]=[#6](-[#6]#[#7])[#6]=[#6][#6]=1',
  '*=[#6](-*)(-[#6]#[#7])',
  '*-[#6]=[#6]-[#6]=[#6]-*',
  '*-[#6]=[#7]-[#6]=[#6]-*',
  '*-[#7]=[#7]-*',
  '*=[#6]-[#6](=[#8])-[#6]=*',
  '[#6](=*)-[#7](-[#6]=*)(-[#6]=*)',
  '*=[#6]-[#6](=[#8])-[#7](-[#6]=*)(-[#6]=*)',
]

for i, smarts in enumerate(smarts_patterns):
  print(i, smarts)
  visualize_smarts_rdkit(smarts)
  #visualize_smarts_smartsplus(smarts)
  print('=' * 50)

### SMARTS for substrates

In [ ]:
smarts_counts_substrate = {}
for i, smarts in enumerate(smarts_patterns):
  print(i, smarts)
  smarts_counts_substrate[f'S{i}'] = visualize_smarts_with_smiles(df_valid_substrates['smiles'].tolist(), smarts)
  value = smarts_counts_substrate[f'S{i}']
  print(f'match count: {value}')
  print('=' * 50)

In [ ]:
sns.barplot(x=list(smarts_counts_substrate.keys()), y=list(smarts_counts_substrate.values()))
plt.show()

### SMARTS for pore side

In [ ]:
smarts_counts_side = {}
for i, smarts in enumerate(smarts_patterns):
  print(i, smarts)
  smarts_counts_side[f'S{i}'] = visualize_smarts_with_smiles(df_valid_poreside['smiles'].tolist(), smarts)
  value = smarts_counts_side[f'S{i}']
  print(f'match count: {value}')
  print('=' * 50)

In [ ]:
sns.barplot(x=list(smarts_counts_side.keys()), y=list(smarts_counts_side.values()))
plt.show()

### SMARTS for molecule with a node

In [ ]:
smarts_counts_node = {}
for i, smarts in enumerate(smarts_patterns):
  print(i, smarts)
  smarts_counts_node[f'S{i}'] = visualize_smarts_with_smiles(df_valid_node['smiles'].tolist(), smarts)
  value = smarts_counts_node[f'S{i}']
  print(f'match count: {value}')
  print('=' * 50)

In [ ]:
sns.barplot(x=list(smarts_counts_node.keys()), y=list(smarts_counts_node.values()))
plt.show()

In [ ]:
df_smarts_substrate = pd.DataFrame(list(smarts_counts_substrate.items()), columns=['smarts', 'count'])
df_smarts_poreside = pd.DataFrame(list(smarts_counts_side.items()), columns=['smarts', 'count'])
df_smarts_node = pd.DataFrame(list(smarts_counts_node.items()), columns=['smarts', 'count'])

df_smarts_substrate['type'] = 'substrate'
df_smarts_poreside['type'] = 'poreside'
df_smarts_node['type'] = 'node'

df_smarts = pd.concat([df_smarts_substrate, df_smarts_poreside, df_smarts_node])

sns.barplot(x='smarts', y='count', hue='type', data=df_smarts)
plt.xticks(rotation=90)
plt.show()